<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# M07 Low Code ML - EDA

---
<small>© 2026 Decoded Limited. All rights reserved. Website: https://decoded.com </small>

### Rapid Data Summarisation with ydata-profiling and Pandas_dq

 Being able to visualise and summarise your data is essential for ML steps such Exploratory Data Analysis (EDA) and Data Preprocessing. However, writing individual lines of code to check distributions, correlations, and outliers for *every single column* can be quite overwhelming and stall your development of an initial ML model.

- with **ydata-profiling**. In just a few lines of code, it generates a comprehensive, interactive HTML report of your dataset.
- with **pandas_dq** in just a few lines of code, provides a list of data quality issues and a how to fix them

In this notebook, we will use it to perform rapid EDA on the **Juice** (Classification) and **Insurance** (Regression)

In [ ]:
# Import libraries
%matplotlib inline
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pycaret.datasets import get_data
from ydata_profiling import ProfileReport
from pandas_dq import dq_report,Fix_DQ
from sklearn.model_selection import train_test_split

## 0. Loading The Data From A SQLlite Database

#### Connecting the database

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("data.db")

#### What does this database contain?

In [ ]:
# what tables does this
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';",conn)
tables

Now we know the tables found in this database. Let's investigate more about the table

## 1. Loading the Data


In [ ]:
### Import all rows from the table using read_sql in pandas
juice_data = pd.read_sql("SELECT * FROM purchases", conn)
df = juice_data.copy()

In [ ]:
#Load Classification Data
juice_data.head()

In [ ]:
#Citrus Hill and Minute Maid
juice_data["Purchase"].value_counts()

## 2. Generating the Profile Reports
Instead of writing dozens of `df.describe()` and `df.hist()` commands, we pass our dataframes into `ProfileReport`.

*Note: To save memory in the notebook, we will export the report as an HTML file.*

Now that we have alleviated the need for extensive plots we can now focus on the decision making needed for modelling before this let us just quickly check the data

# 3. Assessing Data Quality Issues with Pandas_DQ

whilst many data quality issues are flagged in the alerts pane of the report, it does not recommend the suggested fixes, if you prefer to have those hints you may opt to use Pandas_dq too

In [ ]:
dq_report(juice_data,verbose=0)

### 0. Data Familiarization (Sanity Check)

What is the shape of the data?
→ Determines scale and modeling constraints
→ ✅ Choose model complexity, validation strategy

In [ ]:
# Shape of data (rows, columns)
juice_data.shape

What does a sample look like?
→ Reveals structure and obvious issues
→ ✅ Identify target, IDs, feature types

In [ ]:
juice_data.head()

Are data types correct?
→ Affects all downstream processing
→ ✅ Fix dtypes, separate feature groups

In [ ]:
# Data types
juice_data.dtypes

Any obvious anomalies?
→ Early detection of data issues
→ ✅ Flag problems for deeper analysis later

In [ ]:
# Data types
juice_data.describe()

## EDA informed Considerations

### 1. Look at the Target Variable

Binary or multi-class?
- Changes model type and evaluation approach

Missing target values?

- Invalid training samples
- ✅ Drop or fix before modeling

In [ ]:
# 4. Check missing target values (code)
juice_data["Purchase"].isnull().sum()

Which metric reflects business goal?
- Accuracy can be misleading
- ✅ Choose F1, recall, precision, ROC-AUC, PR-AUC

### 2. Data Quality

Shape of Data?
- how many rows do we have?
- how many columns?

In [ ]:
juice_data.shape

Incorrect data types?
→ Misleads models (e.g., numeric treated as categorical)
→ ✅ Fix dtype → impacts encoding + model input

In [ ]:
juice_data.dtypes

→ Missingness can carry signal or introduce bias*
→ ✅ Choose imputation strategy (mean, median, model-based, or flag)

In [ ]:
# Missing values (proportion)
juice_data.isnull().sum().sort_values(ascending=False)

Are there constant / near-constant columns?
→ No predictive value

In [ ]:
constant_cols = [col for col in juice_data.columns if juice_data[col].nunique() <= 1]

Duplicate rows?
→ Can bias model and inflate performance
→ ✅ Remove duplicates

In [ ]:
# Count duplicates
juice_data.duplicated().sum()

In [ ]:
# Drop duplicates
juice_data = juice_data.drop_duplicates()

- we can drop duplicates before train/test split

In [ ]:
juice_data.shape

High-cardinality categorical features?***
→ Can explode feature space
→ ✅ Choose encoding (target encoding, embeddings)

In [ ]:
# Number of unique values per column
juice_data.nunique().sort_values().head()

### 3. Feature Distributions

Are numeric features skewed?

In [ ]:
# Skewness of numeric features
juice_data.skew(numeric_only=True).sort_values(ascending=False)

Are numeric features on very different scales?
→ Some models are sensitive to magnitude, so large-scale variables can dominate learning

- based on this when we do our train test split we may expect to applying a transform later

In [ ]:
juice_data.describe().T[["min", "max"]]

Are categorical variables dominated by few values?*

→ Low information content


In [ ]:
for col in juice_data.select_dtypes(include='object').columns:
    print(f"\nColumn: {col}")
    print(juice_data[col].value_counts(normalize=True).head())

- remember not to drop

### 4. Feature vs Target Relationships (Signal Detection)

**Do features separate classes clearly?**
→ Indicates strong predictive power → ✅ Prioritize these features *

In [ ]:
juice_data.groupby("Purchase").mean(numeric_only=True).T

In [ ]:
# Correlation with target (if binary: 0/1)
juice_data["Purchase_binary"] = juice_data["Purchase"].map({
    "CH": 1,
    "MM": 0
})

**No visible relationship?**
→ Likely noise → ✅ Drop or deprioritize *

In [ ]:
juice_data.corr(numeric_only=True)["Purchase_binary"].sort_values()

### 5. Correlation & Redundancy (Between Features)

Highly correlated features?

→ Redundant information
→ ✅ Drop one (simpler model, avoids multicollinearity)

In [ ]:
corr = juice_data.corr(numeric_only=True).abs()
corr[corr > 0.8]

- lets make this abit better

In [ ]:
corr = juice_data.corr(numeric_only=True)

# Mask lower triangle (including diagonal)
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix (Upper Triangle Only)")
plt.show()

- Remember not to drop columns just yet just loosely be aware they maybe be a possibility of dropping some collinear columns later in the train,test split.

### 6. Outliers & Anomalies

Are outliers real or errors?

- Errors hurt model, real ones may be important
- ✅ Remove errors, keep meaningful extremes

In [ ]:
# IQR-based outlier count per column
numeric_data = juice_data.select_dtypes(include='number')

Q1 = numeric_data.quantile(0.25,numeric_only=True)
Q3 = numeric_data.quantile(0.75,numeric_only=True)
IQR = Q3 - Q1

outliers = ((numeric_data < (Q1 - 1.5 * IQR)) | (numeric_data > (Q3 + 1.5 * IQR))).sum()
outliers.sort_values(ascending=False)

✅ Transform (log, sqrt)
- if skewed distribution

✅ Leave it (if meaningful)

- high income, large purchases

🟡 Moderate outliers → model-dependent decision

Tree models → usually fine

Linear models → more sensitive

# Using Fix DQ

In [ ]:
clean_df = Fix_DQ().fit_transform(df)

In [ ]:
dq_report(clean_df)

# Summary

we've used pandas profiling and pandas_dq to help us understand or data, and did the manual code version to verify/better illustrate our findings

In [ ]:
juice_data

# Low code ML wih Pycaret

## Interpreting The Preprocessing Steps From Output

Look at the orignal data shape and the transformed data shape:

how many more columns were added?
which transformation step increased the columns count?:
  
- scaling
- label_encoding
- one hot encoding
  

Look at the imputation types used to handle missing values:
- For missing numerical features which imputation method was used?

Look at the categorical imputation types

which case does the mean not make sense:
- if the column contains outliers the mean can be distorted
- we should default to the median instead always
- we use the median imputation instead if outliers are present

Stratified K folds was used, what is the benefit of this? in the classification context?

# Interpreting The Recall And Precision

For your best model and metric of focus:

let Y be your percentage recall and X the percentage precision, fill in the summary statment about your model output

We detect Y% of true cases Salary with X% confidence in each alert

Discuss how this is relevant to your business problem context

# Hyperparameter tuning

suppose you want to improve your model with respect to precision,recall or f1 you use Pycaret hyperparameter tuning like in the below commented out code (outputs alot of print statements be careful)

In [ ]:
#Tune the hyperparameters of the best model

# tuned_model = tune_model(best_model,optimize='recall')
# print(tuned_model)

# Evaluate On Test Set

lets get the metrics for the evaluation on the test set

In [ ]:
# use best_model if you didn't run the cell above


# Confusion Matrix


now that we've interpreted the recall and precision values as percentages:

let us more describe more concretely what has happened using the confusion matrix



In [ ]:
# use best_model if you didn't run the cell above


Fill in the statements below to evaluate the confusion matrix:

- The count of True Positives (predicted class 1 and is actually class 1) is _______
- The count of True Negative  (predicted class 0 and is actually class 0) is _______
- Of the _____________ Total Positive predictions _______________ of out of  _______ were correct
- Of the _____________ Total Negative predictions _______________ of out of  _______ were correct

# Saving your model

In [ ]:
# finalize for production


In [ ]:

# save
